# TripoSG — prop generator (free Colab T4)

Generates ONE watertight 3D prop from a single reference image, using **TripoSG** (VAST-AI-Research).

- **Licence:** MIT code + MIT weights — the cleanest licence in AI 3D.
- **GPU:** 8 GB VRAM — a free Colab T4 (16 GB) runs it comfortably. This is the T4-safe heavy tool.
- **Honest limits:** image-only (no text-to-3D) and **geometry-only** (no texture, no UVs — paint the material in Blender/Godot). Output is a watertight manifold mesh, **Y-up** — rotate to Z-up before Godot import.
- **Props only, never the hero, never a face.** One good prop per 10–20 min of cloud time + 30–60 min of Blender cleanup.

A rock-specific run of this same pipeline is already in progress for the golem-fight clearing: `workspace/cloud-sync/heavy-tools/triposg-prop-generator.ipynb` (mossy rock). This notebook is the general version for any prop.

Steps: 1) check GPU, 2) clone + install, 3) upload image + inference → `prop.glb`, 4) rotate to Z-up + decimate, 5) verify outputs.

In [ ]:
# 1) GPU check — expect a T4 with ~16 GB
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# 2) Clone TripoSG + install (weights ~6 GB download on first use)
%cd /content
!git clone https://github.com/VAST-AI-Research/TripoSG.git
%cd /content/TripoSG
!pip install -r requirements.txt -q
print('install done')

In [ ]:
# 3) Inference: upload ONE reference image -> prop.glb
from google.colab import files
print('Upload one reference image (PNG/JPG, object on a plain background):')
up = files.upload()
img_path = '/content/' + list(up.keys())[0]
print('input:', img_path)

%cd /content/TripoSG
import torch
from PIL import Image
from triposg import TripoSG
from triposg.utils import remove_background

model = TripoSG.from_pretrained('VAST-AI/TripoSG')
model.cuda()
image = Image.open(img_path).convert('RGBA')
image = remove_background(image)
with torch.no_grad():
    mesh = model(image, num_inference_steps=50)[0]
mesh.export('/content/prop.glb', postprocess=True)
print('exported /content/prop.glb')

In [ ]:
# 4) Rotate Y-up -> Z-up (Godot) + decimate to ~10k faces (guarded)
import os
try:
    import trimesh
    import numpy as np
    m = trimesh.load('/content/prop.glb', force='mesh')
    print('faces before:', len(m.faces))
    # TripoSG exports Y-up; Godot is Z-up: rotate -90 deg about X
    rot = trimesh.transformations.rotation_matrix(np.radians(-90), [1, 0, 0])
    m.apply_transform(rot)
    if len(m.faces) > 10000:
        m = m.simplify_quadric_decimation(10000)
        print('faces after decimation:', len(m.faces))
    m.export('/content/prop-zup-10k.glb')
    print('exported /content/prop-zup-10k.glb')
except Exception as e:
    print('postprocess skipped:', e)
    print('NOTE: /content/prop.glb is still valid — do the Z-up rotation in Blender instead')

In [ ]:
# 5) Verify outputs
import os
for f in ['/content/prop.glb', '/content/prop-zup-10k.glb']:
    print(f, os.path.getsize(f) if os.path.exists(f) else 'MISSING', 'bytes')
!ls -la /content/*.glb